# 🌍 Synthetic Seismic Data Generation — Professional Notebook
### Geophysical Forward Modeling: Faults · Fractures · Folds · Unconformities · Salt Bodies

---

## 1. Theoretical Background

### 1.1 Seismic Reflection Principle

Seismic reflection surveys record **compressional (P-wave)** or **shear (S-wave)** energy that bounces off subsurface impedance contrasts. The fundamental measurement is **two-way travel time (TWT)** — the time for a wave to travel from surface to a reflector and back.

**Acoustic impedance (Z)** is the product of bulk density (ρ) and P-wave velocity (Vp):

$$Z = \rho \cdot V_p \quad [\text{kg/m}^2\text{s}]$$

At each geological interface, the **reflection coefficient (RC)** determines what fraction of energy is reflected:

$$RC = \frac{Z_2 - Z_1}{Z_2 + Z_1}$$

where $Z_1$ and $Z_2$ are the impedances above and below the interface.

---

### 1.2 Convolutional Model of Seismogram

The observed seismic trace $s(t)$ is the convolution of a source wavelet $w(t)$ with the subsurface reflectivity series $r(t)$, plus noise $n(t)$:

$$s(t) = w(t) * r(t) + n(t)$$

This **convolutional model** assumes:
- Plane-wave propagation (1-D earth)
- Primary reflections only (no multiples in simplified model)
- Linear, time-invariant system
- Stationary wavelet

---

### 1.3 Source Wavelets

**Ricker Wavelet** (second derivative of Gaussian — zero phase, symmetric):

$$w(t) = \left(1 - 2\pi^2 f_0^2 t^2\right) e^{-\pi^2 f_0^2 t^2}$$

**Ormsby Wavelet** (bandpass, defined by four frequencies $f_1 < f_2 < f_3 < f_4$):

$$w(f) = \begin{cases} 0 & f < f_1 \\ \text{linear ramp} & f_1 \leq f < f_2 \\ 1 & f_2 \leq f \leq f_3 \\ \text{linear ramp} & f_3 < f \leq f_4 \\ 0 & f > f_4 \end{cases}$$

---

### 1.4 Geological Structures Modeled

| Structure | Physical Mechanism | Seismic Expression |
|---|---|---|
| **Horizontal Layers** | Depositional strata | Continuous sub-horizontal reflectors |
| **Anticline / Syncline** | Compressional folding | Curved/bowed reflectors, structural highs/lows |
| **Normal Fault** | Extensional tectonics | Offset reflectors, fault-plane reflection, pull-apart |
| **Reverse Fault** | Compressional tectonics | Overlapping/stacked reflectors, thrust ramps |
| **Fracture Zones** | Local brittle failure | Velocity anomalies, chaotic/dim zones, attenuation |
| **Unconformity** | Erosional surface | Angular truncation of reflectors |
| **Salt / Diapir** | Buoyancy-driven flow | Chaotic interior, strong top/base reflections, velocity pull-up |
| **Pinch-Out / Wedge** | Onlap / offlap | Reflectors that thin to termination |

---

### 1.5 Noise Models in Seismic Data

- **Gaussian (white) noise** — random electronic/ambient noise, flat power spectrum
- **Random noise** — environmental (wind, traffic, water)
- **Coherent noise** — surface waves (ground roll), direct arrival, multiples
- **Signal-to-Noise Ratio (SNR)**: $\text{SNR} = 20 \log_{10}\left(\frac{A_{\text{signal}}}{A_{\text{noise}}}\right)$ dB

---

In [ ]:
# ============================================================
#  CELL 1 — Imports & Global Configuration
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from scipy.signal import convolve, butter, filtfilt
from scipy.ndimage import gaussian_filter, uniform_filter
from scipy.interpolate import interp1d
from matplotlib.patches import FancyArrowPatch
from matplotlib.ticker import MultipleLocator
import warnings
warnings.filterwarnings('ignore')

# ── Plotting style ─────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'figure.dpi'       : 120,
    'axes.grid'        : False,
    'image.interpolation': 'bilinear',
})

RNG = np.random.default_rng(seed=42)   # reproducible noise

# ── Survey geometry ────────────────────────────────────────
N_TRACES   = 300      # number of CMP traces (x-axis)
N_SAMPLES  = 500      # time samples per trace
DT         = 0.002    # sample interval [s]  → 2 ms
DX         = 25.0     # trace spacing [m]
DEPTH_MAX  = 3000     # maximum model depth [m]

time_axis  = np.arange(N_SAMPLES) * DT          # TWT [s]
x_axis     = np.arange(N_TRACES)  * DX          # distance [m]
TMAX       = time_axis[-1]

print(f"Survey parameters:")
print(f"  Traces         : {N_TRACES}")
print(f"  Time samples   : {N_SAMPLES}  (dt = {DT*1000:.1f} ms)")
print(f"  Max TWT        : {TMAX:.3f} s")
print(f"  Trace spacing  : {DX} m")
print(f"  Total profile  : {x_axis[-1]/1000:.2f} km")

---
## 2. Wavelet Design

We implement three industry-standard wavelets. The **Ricker** is the most widely used in synthetic modeling. The **Ormsby** better mimics real bandpass acquisition. **Klauder** wavelets arise from vibroseis sweep correlation.

In [ ]:
# ============================================================
#  CELL 2 — Wavelet Factory
# ============================================================

def ricker_wavelet(f0: float, dt: float, duration: float = 0.128) -> np.ndarray:
    """Zero-phase Ricker (Mexican-hat) wavelet.
    
    Parameters
    ----------
    f0       : dominant (peak) frequency [Hz]
    dt       : sample interval [s]
    duration : total length of wavelet [s]
    """
    t  = np.arange(-duration/2, duration/2, dt)
    u  = (np.pi * f0 * t) ** 2
    w  = (1 - 2*u) * np.exp(-u)
    return t, w / np.max(np.abs(w))


def ormsby_wavelet(f1, f2, f3, f4, dt: float, duration: float = 0.128) -> np.ndarray:
    """Zero-phase Ormsby bandpass wavelet defined by 4 corner frequencies."""
    def A(f, fa, fb):
        """Ramp function for Ormsby."""
        return (np.sinc(fb*(t-1e-10)) - np.sinc(fa*(t-1e-10))) * (fa**2 / (fa**2 - fb**2))
    
    t  = np.arange(-duration/2, duration/2 + dt, dt)
    w  = ((np.pi*f4)**2 / (np.pi*f4 - np.pi*f3)) * np.sinc(f4*t)**2 \
       - ((np.pi*f3)**2 / (np.pi*f4 - np.pi*f3)) * np.sinc(f3*t)**2 \
       - ((np.pi*f2)**2 / (np.pi*f2 - np.pi*f1)) * np.sinc(f2*t)**2 \
       + ((np.pi*f1)**2 / (np.pi*f2 - np.pi*f1)) * np.sinc(f1*t)**2
    w /= np.max(np.abs(w))
    return t, w


def klauder_wavelet(f_low, f_high, dt, T_sweep=10.0, duration=0.128):
    """Klauder wavelet — autocorrelation of a linear vibroseis sweep."""
    t_full = np.arange(0, T_sweep, dt)
    sweep  = np.cos(2*np.pi * (f_low*t_full + 0.5*(f_high-f_low)/T_sweep * t_full**2))
    t_wav  = np.arange(-duration/2, duration/2, dt)
    n_half = len(t_wav)//2
    corr   = np.correlate(sweep, sweep, mode='full')
    mid    = len(corr) // 2
    w      = corr[mid - n_half : mid + n_half]
    if len(w) > len(t_wav): w = w[:len(t_wav)]
    w /= np.max(np.abs(w))
    return t_wav[:len(w)], w


# ── Generate wavelets ──────────────────────────────────────
t_rick, w_rick   = ricker_wavelet(f0=35, dt=DT)
t_orm,  w_orm    = ormsby_wavelet(5, 10, 60, 80, dt=DT)
t_klaud, w_klaud = klauder_wavelet(10, 80, dt=DT)

# ── Plot wavelets ──────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
wavelet_data = [
    (t_rick,  w_rick,  'Ricker (35 Hz)',   '#1f77b4'),
    (t_orm,   w_orm,   'Ormsby (5-10-60-80 Hz)', '#ff7f0e'),
    (t_klaud, w_klaud, 'Klauder (10-80 Hz)', '#2ca02c'),
]

for col, (t_w, w, label, clr) in enumerate(wavelet_data):
    # Time domain
    ax = axes[0, col]
    ax.plot(t_w*1000, w, color=clr, lw=2)
    ax.fill_between(t_w*1000, w, 0, where=(w>0), alpha=0.3, color=clr)
    ax.fill_between(t_w*1000, w, 0, where=(w<0), alpha=0.3, color='gray')
    ax.axhline(0, color='k', lw=0.7, ls='--')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Time [ms]')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(t_w[0]*1000, t_w[-1]*1000)

    # Amplitude spectrum
    ax2 = axes[1, col]
    nfft = 512
    spec = np.abs(np.fft.rfft(w, n=nfft))
    freq = np.fft.rfftfreq(nfft, d=DT)
    ax2.plot(freq, spec, color=clr, lw=2)
    ax2.fill_between(freq, spec, alpha=0.25, color=clr)
    ax2.set_xlim(0, 150)
    ax2.set_xlabel('Frequency [Hz]')
    ax2.set_ylabel('|Amplitude|')
    ax2.set_title(f'Amplitude Spectrum — {label.split("(")[0].strip()}')

fig.suptitle('Seismic Source Wavelets: Time Domain & Frequency Spectra',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('wavelets.png', bbox_inches='tight', dpi=150)
plt.show()
print("Wavelets generated.")

---
## 3. Rock Physics & Elastic Properties

We define a realistic stratigraphic column with representative **Vp, Vs, and density (ρ)** for common lithologies, following Castagna & Backus (1993) and the AAPG Handbook of Physical Properties.

In [ ]:
# ============================================================
#  CELL 3 — Rock Physics Database
# ============================================================

# Each layer: (name, Vp [m/s], Vs [m/s], rho [kg/m3], color)
ROCK_DB = {
    'Water'          : dict(Vp=1480,  Vs=0,    rho=1020, color='#a8d8f0'),
    'Unconsolidated' : dict(Vp=1700,  Vs=600,  rho=1800, color='#f5e6c8'),
    'Shale_shallow'  : dict(Vp=2200,  Vs=900,  rho=2200, color='#8B7355'),
    'Sand_dry'       : dict(Vp=2400,  Vs=1500, rho=2100, color='#F4D03F'),
    'Sand_brine'     : dict(Vp=2600,  Vs=1400, rho=2150, color='#F8C471'),
    'Sand_gas'       : dict(Vp=2100,  Vs=1450, rho=2050, color='#F0E68C'),
    'Limestone'      : dict(Vp=4500,  Vs=2500, rho=2600, color='#C0C0C0'),
    'Dolomite'       : dict(Vp=5500,  Vs=3000, rho=2800, color='#A9A9A9'),
    'Chalk'          : dict(Vp=3000,  Vs=1500, rho=2300, color='#FFFACD'),
    'Shale_deep'     : dict(Vp=3200,  Vs=1500, rho=2450, color='#696969'),
    'Coal'           : dict(Vp=2200,  Vs=1100, rho=1350, color='#2C2C2C'),
    'Anhydrite'      : dict(Vp=6100,  Vs=3100, rho=2960, color='#DDA0DD'),
    'Salt'           : dict(Vp=4480,  Vs=2590, rho=2160, color='#FFB6C1'),
    'Granite'        : dict(Vp=5950,  Vs=3400, rho=2670, color='#CD853F'),
    'Basalt'         : dict(Vp=5500,  Vs=3200, rho=2900, color='#4A4A4A'),
    'Fracture_zone'  : dict(Vp=1900,  Vs=800,  rho=2000, color='#FF6347'),
}

def acoustic_impedance(rock_name):
    r = ROCK_DB[rock_name]
    return r['Vp'] * r['rho']

def reflection_coeff(rock_above, rock_below):
    Z1 = acoustic_impedance(rock_above)
    Z2 = acoustic_impedance(rock_below)
    return (Z2 - Z1) / (Z2 + Z1)

# ── Print RC table ─────────────────────────────────────────
print(f"{'Interface':<35}  {'Z1':>10}  {'Z2':>10}  {'RC':>8}")
print("-" * 70)
interfaces = [
    ('Unconsolidated', 'Shale_shallow'),
    ('Shale_shallow',  'Sand_brine'),
    ('Shale_shallow',  'Sand_gas'),
    ('Shale_shallow',  'Limestone'),
    ('Sand_brine',     'Salt'),
    ('Shale_deep',     'Limestone'),
    ('Limestone',      'Dolomite'),
    ('Shale_deep',     'Granite'),
]
for r1, r2 in interfaces:
    Z1 = acoustic_impedance(r1)
    Z2 = acoustic_impedance(r2)
    rc = reflection_coeff(r1, r2)
    tag = " ← bright" if abs(rc) > 0.12 else ""
    print(f"{r1+' / '+r2:<35}  {Z1:>10,}  {Z2:>10,}  {rc:>+8.4f}{tag}")

# ── Impedance bar chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
names = list(ROCK_DB.keys())
imps  = [acoustic_impedance(n) for n in names]
colors = [ROCK_DB[n]['color'] for n in names]
bars = ax.bar(names, imps, color=colors, edgecolor='k', linewidth=0.6)
ax.set_ylabel('Acoustic Impedance [kg/m²s]', fontsize=11)
ax.set_title('Acoustic Impedance by Lithology', fontweight='bold')
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f} M'))
plt.tight_layout()
plt.savefig('impedance.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 4. Geological Model Construction

We build a **2-D geological model** in *sample space* (trace × time-sample). Each cell stores a lithology index. We then progressively add:

1. Background layered stratigraphy
2. Anticline fold
3. Normal fault
4. Reverse (thrust) fault
5. Unconformity surface
6. Salt diapir
7. Fracture zones

In [ ]:
# ============================================================
#  CELL 4 — Build Geological Model (Litho Index Grid)
# ============================================================

# ── Layer sequence (name, base time-sample in reference column) ─
#    We define the stratigraphic column at the CENTER trace, then
#    deform it using structural functions.
LAYERS = [
    ('Unconsolidated', 30),    # 0 – 60 ms  surface
    ('Shale_shallow',  70),    # down to 140 ms
    ('Sand_gas',      100),    # thin gas sand
    ('Shale_shallow', 140),    # inter-bedded shale
    ('Sand_brine',    170),    # brine sand
    ('Shale_shallow', 210),
    ('Coal',          215),    # thin coal seam
    ('Shale_shallow', 250),
    ('Chalk',         295),
    ('Shale_deep',    320),
    ('Limestone',     370),
    ('Dolomite',      400),
    ('Shale_deep',    430),
    ('Anhydrite',     450),
    ('Shale_deep',    480),
    ('Granite',       500),    # basement — fills to N_SAMPLES
]

LITHO_NAMES = list(ROCK_DB.keys())

def litho_idx(name):
    return LITHO_NAMES.index(name)

# ── Helper: time-shift map for a Gaussian fold ──────────────
def gaussian_fold(x, x_centre, amplitude, sigma):
    """Returns per-trace time-sample shift (positive = pushed deeper)."""
    return amplitude * np.exp(-((x - x_centre)**2) / (2*sigma**2))

# ── Pre-compute structural deformation maps ─────────────────
xi = np.arange(N_TRACES, dtype=float)

# Anticline (upward arch, negative shift)
anticline_shift = -gaussian_fold(xi, x_centre=80,  amplitude=40, sigma=22)
# Syncline
syncline_shift  = +gaussian_fold(xi, x_centre=220, amplitude=25, sigma=18)
# Regional dip (left side deeper by ~15 samples)
dip_shift       = 0.05 * xi

total_fold_shift = (anticline_shift + syncline_shift + dip_shift).astype(int)

# ── Build base litho grid (before faulting) ─────────────────
litho_grid = np.zeros((N_TRACES, N_SAMPLES), dtype=np.int8)

for tr in range(N_TRACES):
    shift = total_fold_shift[tr]
    sample_ptr = 0
    prev_base  = 0
    for (lname, base_samp) in LAYERS:
        deformed_base = int(np.clip(base_samp + shift, 0, N_SAMPLES - 1))
        litho_grid[tr, prev_base:deformed_base] = litho_idx(lname)
        prev_base = deformed_base
    litho_grid[tr, prev_base:] = litho_idx('Granite')

# ── Normal Fault ─────────────────────────────────────────────
# Fault plane from trace 130 dipping at ~70° (listric), throw = 22 samples
NORMAL_FAULT_TRACE = 130
NORMAL_FAULT_THROW = 22   # samples, hanging-wall drops DOWN

for tr in range(NORMAL_FAULT_TRACE, N_TRACES):
    # Apply downward throw to hanging wall
    dist_from_fault = tr - NORMAL_FAULT_TRACE
    throw = min(NORMAL_FAULT_THROW, 22)  # uniform throw
    col = litho_grid[tr].copy()
    # Shift samples down
    shifted = np.zeros_like(col)
    shifted[throw:] = col[:N_SAMPLES - throw]
    shifted[:throw] = litho_idx('Shale_shallow')  # fault infill
    litho_grid[tr] = shifted

# ── Reverse / Thrust Fault ──────────────────────────────────
# Fault at trace 200, foot-wall pushed UP by 15 samples
REVERSE_FAULT_TRACE = 200
REVERSE_FAULT_THROW = 15

for tr in range(REVERSE_FAULT_TRACE, min(REVERSE_FAULT_TRACE + 80, N_TRACES)):
    col = litho_grid[tr].copy()
    shifted = np.zeros_like(col)
    shifted[:N_SAMPLES - REVERSE_FAULT_THROW] = col[REVERSE_FAULT_THROW:]
    shifted[N_SAMPLES - REVERSE_FAULT_THROW:] = litho_idx('Granite')
    litho_grid[tr] = shifted

# ── Salt Diapir ─────────────────────────────────────────────
# Elliptical salt body centred at trace=255, sample=220
SALT_CX, SALT_CY = 255, 215
SALT_RX, SALT_RY = 22, 80   # semi-axes in trace / sample units

for tr in range(N_TRACES):
    for samp in range(N_SAMPLES):
        if ((tr - SALT_CX)/SALT_RX)**2 + ((samp - SALT_CY)/SALT_RY)**2 <= 1.0:
            litho_grid[tr, samp] = litho_idx('Salt')

# ── Fracture Zones ───────────────────────────────────────────
# Two sub-vertical fracture zones (width=3 traces)
FRACTURE_ZONES = [60, 175]

for fz in FRACTURE_ZONES:
    for tr in range(max(0, fz-2), min(N_TRACES, fz+2)):
        # Fractures only in deeper part
        litho_grid[tr, 150:420] = litho_idx('Fracture_zone')

# ── Unconformity surface ─────────────────────────────────────
# Erosional surface: truncate layers above sample 160-180 (left side)
for tr in range(0, 90):
    unc_samp = int(165 + 0.2 * tr + total_fold_shift[tr])
    unc_samp = int(np.clip(unc_samp, 100, N_SAMPLES - 1))
    litho_grid[tr, :unc_samp] = litho_idx('Shale_shallow')   # eroded fill

print("Geological model built.")
print(f"  Grid shape: {litho_grid.shape}  (traces × samples)")
print(f"  Unique lithologies present: {np.unique(litho_grid)}")

In [ ]:
# ============================================================
#  CELL 5 — Visualise Geological Model
# ============================================================

# Build a discrete colormap matching litho index order
cmap_colors = [ROCK_DB[n]['color'] for n in LITHO_NAMES]
litho_cmap  = mcolors.ListedColormap(cmap_colors)
bounds      = np.arange(-0.5, len(LITHO_NAMES) + 0.5, 1)
norm        = mcolors.BoundaryNorm(bounds, litho_cmap.N)

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(
    litho_grid.T,           # transpose → time on Y-axis
    aspect='auto',
    cmap=litho_cmap,
    norm=norm,
    origin='upper',
    extent=[0, x_axis[-1]/1000, TMAX*1000, 0]
)

# Colorbar with litho labels
cbar = fig.colorbar(im, ax=ax, pad=0.01, shrink=0.85)
cbar.set_ticks(range(len(LITHO_NAMES)))
cbar.set_ticklabels(LITHO_NAMES, fontsize=7)
cbar.set_label('Lithology', fontsize=10)

# Annotation arrows
annotations = [
    (80*DX/1000,  0.28, 'Anticline',    '#1a1aff'),
    (220*DX/1000, 0.48, 'Syncline',     '#cc0000'),
    (130*DX/1000, 0.65, 'Normal Fault', '#ff6600'),
    (200*DX/1000, 0.60, 'Thrust Fault', '#9900cc'),
    (255*DX/1000, 0.42, 'Salt Diapir',  '#cc0066'),
    (60*DX/1000,  0.72, 'Fracture',     '#008000'),
    (30*DX/1000,  0.35, 'Unconformity', '#994400'),
]
for xpos, ypos, label, clr in annotations:
    ax.annotate(label, xy=(xpos, ypos),
                fontsize=9, fontweight='bold', color=clr,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=clr, alpha=0.8))

ax.set_xlabel('Distance [km]', fontsize=12)
ax.set_ylabel('Two-Way Travel Time [ms]', fontsize=12)
ax.set_title('2-D Geological Model — Synthetic Earth Section', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(MultipleLocator(1))
ax.yaxis.set_major_locator(MultipleLocator(100))

plt.tight_layout()
plt.savefig('geological_model.png', bbox_inches='tight', dpi=150)
plt.show()
print("Geological model visualised.")

---
## 5. Velocity & Impedance Models

From the lithology grid we extract **P-wave velocity (Vp)** and **acoustic impedance (AI)** models. These are the physical quantities needed for forward modeling.

In [ ]:
# ============================================================
#  CELL 6 — Extract Vp, Vs, Rho, AI grids
# ============================================================

Vp_arr  = np.array([ROCK_DB[n]['Vp']  for n in LITHO_NAMES], dtype=float)
rho_arr = np.array([ROCK_DB[n]['rho'] for n in LITHO_NAMES], dtype=float)
Vs_arr  = np.array([ROCK_DB[n]['Vs']  for n in LITHO_NAMES], dtype=float)

Vp_grid  = Vp_arr[litho_grid]    # shape: (N_TRACES, N_SAMPLES)
rho_grid = rho_arr[litho_grid]
Vs_grid  = Vs_arr[litho_grid]
AI_grid  = Vp_grid * rho_grid

# Add mild random heterogeneity (±3%) — geological scatter
heterogeneity = 1.0 + 0.03 * RNG.standard_normal(Vp_grid.shape)
Vp_grid  = Vp_grid  * heterogeneity
rho_grid = rho_grid * (1.0 + 0.01 * RNG.standard_normal(rho_grid.shape))
AI_grid  = Vp_grid * rho_grid

# ── Visualise Vp and AI ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=True)

extent = [0, x_axis[-1]/1000, TMAX*1000, 0]

im0 = axes[0].imshow(Vp_grid.T,  aspect='auto', cmap='jet', origin='upper', extent=extent)
plt.colorbar(im0, ax=axes[0]).set_label('Vp [m/s]')
axes[0].set_title('P-wave Velocity Model  (Vp)', fontweight='bold')
axes[0].set_ylabel('TWT [ms]')
axes[0].set_xlabel('Distance [km]')

im1 = axes[1].imshow(AI_grid.T/1e6, aspect='auto', cmap='RdBu_r', origin='upper', extent=extent)
plt.colorbar(im1, ax=axes[1]).set_label('AI [MRayl]')
axes[1].set_title('Acoustic Impedance Model  (AI = Vp·ρ)', fontweight='bold')
axes[1].set_xlabel('Distance [km]')

for ax in axes:
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(100))

plt.suptitle('Elastic Property Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('velocity_impedance.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 6. Reflectivity Series

The **reflectivity series** $r(t)$ is computed from sample-to-sample impedance differences. We also apply a **transmission-loss correction** — deeper reflectors are weakened by energy losses along the path (spherical divergence + inelastic attenuation).

In [ ]:
# ============================================================
#  CELL 7 — Compute Reflectivity (with Q-attenuation taper)
# ============================================================

# Sample-to-sample impedance contrast
RC_grid = np.zeros_like(AI_grid)
RC_grid[:, 1:] = (AI_grid[:, 1:] - AI_grid[:, :-1]) / \
                 (AI_grid[:, 1:] + AI_grid[:, :-1] + 1e-12)

# ── Transmission-loss (geometric spreading + Q attenuation) ─
#    Approximate: amplitude decays as exp(-alpha*t) * (1/t^0.5)
Q_factor   = 80.0      # quality factor
DOM_FREQ   = 35.0      # dominant frequency for Q calc
alpha      = np.pi * DOM_FREQ / (Q_factor * Vp_grid.mean(axis=0))
t_vec      = time_axis  # TWT
spread_fac = np.where(t_vec > 0, 1.0 / np.sqrt(t_vec + DT), 1.0)
atten_fac  = np.exp(-alpha * t_vec)
geo_atten  = (spread_fac * atten_fac)[np.newaxis, :]  # broadcast over traces

RC_grid_att = RC_grid * geo_atten
# Clip extreme values (AVO anomalies)
RC_grid_att = np.clip(RC_grid_att, -0.6, 0.6)

# ── Plot one representative trace ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 7), sharey=True)
trace_idx  = 80   # anticline trace

axes[0].barh(time_axis*1000, RC_grid[trace_idx],
             height=2, color=np.where(RC_grid[trace_idx] > 0, '#d62728', '#1f77b4'))
axes[0].set_xlabel('RC')
axes[0].set_ylabel('TWT [ms]')
axes[0].set_title(f'Reflectivity — Trace {trace_idx}\n(no attenuation)')
axes[0].invert_yaxis()

axes[1].barh(time_axis*1000, RC_grid_att[trace_idx],
             height=2, color=np.where(RC_grid_att[trace_idx] > 0, '#d62728', '#1f77b4'))
axes[1].set_xlabel('RC (attenuated)')
axes[1].set_title(f'Reflectivity — Trace {trace_idx}\n(geometric + Q attenuation)')

im = axes[2].imshow(RC_grid_att.T, aspect='auto', cmap='seismic',
                    vmin=-0.2, vmax=0.2, origin='upper',
                    extent=[0, x_axis[-1]/1000, TMAX*1000, 0])
plt.colorbar(im, ax=axes[2]).set_label('RC')
axes[2].set_title('2-D Reflectivity Section')
axes[2].set_xlabel('Distance [km]')

plt.suptitle('Seismic Reflectivity Series', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('reflectivity.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 7. Seismic Forward Modeling — Convolution

We convolve each trace's reflectivity series with the **Ricker wavelet** (zero-phase, 35 Hz), then add structured noise. We also apply a simple **NMO stretch** correction for far-offset traces to simulate CMP gather moveout.

In [ ]:
# ============================================================
#  CELL 8 — Convolutional Seismic Modeling
# ============================================================

_, wavelet = ricker_wavelet(f0=35, dt=DT, duration=0.120)

# ── Convolve every trace ─────────────────────────────────────
seismic_clean = np.zeros((N_TRACES, N_SAMPLES))

for tr in range(N_TRACES):
    conv_full = convolve(RC_grid_att[tr], wavelet, mode='full')
    # Keep central N_SAMPLES portion
    start = len(wavelet) // 2
    seismic_clean[tr] = conv_full[start : start + N_SAMPLES]

# ── Noise model ──────────────────────────────────────────────
SNR_DB        = 12.0     # signal-to-noise ratio in dB
signal_rms    = np.sqrt(np.mean(seismic_clean**2))
noise_rms     = signal_rms / (10 ** (SNR_DB / 20.0))

# White Gaussian noise
noise_gauss   = noise_rms * RNG.standard_normal(seismic_clean.shape)

# Coherent noise: low-freq ground roll (slow apparent velocity)
ground_roll   = np.zeros_like(seismic_clean)
for tr in range(N_TRACES):
    delay     = tr // 3
    gr_amp    = noise_rms * 1.8 * np.exp(-time_axis / 0.15)  # dies off with time
    gr_trace  = gr_amp * np.sin(2*np.pi * 8 * (time_axis - delay * DT * 3))
    ground_roll[tr] = gr_trace * np.exp(-tr / 60.0)

seismic_noisy = seismic_clean + noise_gauss + 0.4 * ground_roll

# ── Automatic Gain Control (AGC) for display ─────────────────
def agc(data, window_samples=50):
    """Trace-by-trace AGC to equalise amplitude for display."""
    agc_data = np.zeros_like(data)
    for tr in range(data.shape[0]):
        env = np.abs(data[tr])
        env_smooth = uniform_filter(env, size=window_samples) + 1e-10
        agc_data[tr] = data[tr] / env_smooth
    return agc_data

seismic_agc   = agc(seismic_noisy, window_samples=60)

print(f"Forward modeling complete.")
print(f"  Signal RMS : {signal_rms:.5f}")
print(f"  Noise RMS  : {noise_rms:.5f}")
print(f"  SNR (design): {SNR_DB} dB")

In [ ]:
# ============================================================
#  CELL 9 — Plot Final Seismic Sections
# ============================================================

def plot_seismic_section(data, title, cmap='seismic', vclip=None,
                          wiggle=True, wiggle_traces=60, ax=None,
                          save_name=None):
    """Professional seismic section display with optional wiggles."""
    standalone = (ax is None)
    if standalone:
        fig, ax = plt.subplots(figsize=(16, 6))

    vm = vclip if vclip else np.percentile(np.abs(data), 98)
    extent_km = [0, x_axis[-1]/1000, TMAX*1000, 0]

    ax.imshow(data.T, aspect='auto', cmap=cmap, vmin=-vm, vmax=vm,
              origin='upper', extent=extent_km)

    if wiggle:
        step = max(1, N_TRACES // wiggle_traces)
        scale = (x_axis[-1]/1000) / N_TRACES * step * 1.5 / (vm + 1e-12)
        for tr in range(0, N_TRACES, step):
            x_pos = tr * DX / 1000
            w_scaled = data[tr] * scale
            ax.plot(x_pos + w_scaled, time_axis*1000,
                    'k-', lw=0.3, alpha=0.5)
            ax.fill_betweenx(time_axis*1000,
                             x_pos, x_pos + w_scaled,
                             where=(data[tr] > 0),
                             color='k', alpha=0.35, lw=0)

    ax.set_xlabel('Distance [km]', fontsize=11)
    ax.set_ylabel('TWT [ms]', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(100))
    ax.set_xlim(0, x_axis[-1]/1000)
    ax.set_ylim(TMAX*1000, 0)

    if standalone:
        plt.tight_layout()
        if save_name:
            plt.savefig(save_name, bbox_inches='tight', dpi=150)
        plt.show()


# ── 3-panel comparison ────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 18))

plot_seismic_section(seismic_clean, 'Synthetic Seismic — Clean (no noise, AGC off)',
                     wiggle=True, ax=axes[0])
plot_seismic_section(seismic_noisy, 'Synthetic Seismic — With Noise (Gaussian + Ground Roll)',
                     wiggle=True, ax=axes[1])
plot_seismic_section(seismic_agc,   'Synthetic Seismic — AGC Applied (display balanced)',
                     wiggle=True, ax=axes[2])

plt.suptitle('Synthetic Seismic Sections — Forward Model Output', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('seismic_sections.png', bbox_inches='tight', dpi=150)
plt.show()
print("Seismic sections plotted.")

---
## 8. Seismic Attributes Computation

**Post-stack seismic attributes** are transformations of the seismic trace that highlight geological or stratigraphic features:

| Attribute | Formula | Geological Use |
|---|---|---|
| **Envelope (Instantaneous Amplitude)** | $A(t) = \sqrt{s(t)^2 + \hat{s}(t)^2}$ | Bright spots, DHI detection |
| **Instantaneous Phase** | $\phi(t) = \arctan\left(\hat{s}(t)/s(t)\right)$ | Continuity, faults |
| **Instantaneous Frequency** | $f_i(t) = \frac{1}{2\pi}\frac{d\phi}{dt}$ | Lithology, fluid |
| **Cosine of Inst. Phase** | $\cos(\phi)$ | Lateral continuity of reflectors |
| **RMS Amplitude** | $\sqrt{\frac{1}{N}\sum s_i^2}$ | Hydrocarbon fairway mapping |
| **Semblance/Coherence** | Multi-trace correlation | Fault and fracture detection |

In [ ]:
# ============================================================
#  CELL 10 — Seismic Attribute Analysis
# ============================================================

from scipy.signal import hilbert

def compute_attributes(seis_data):
    """Compute standard seismic attributes via Hilbert transform."""
    attrs = {}
    analytic = hilbert(seis_data, axis=1)   # analytic signal per trace

    attrs['envelope']       = np.abs(analytic)
    attrs['inst_phase']     = np.angle(analytic)      # radians
    attrs['cos_inst_phase'] = np.cos(attrs['inst_phase'])

    # Instantaneous frequency
    phase_unwrapped  = np.unwrap(attrs['inst_phase'], axis=1)
    attrs['inst_freq'] = np.abs(np.gradient(phase_unwrapped, DT, axis=1) / (2*np.pi))
    attrs['inst_freq']  = np.clip(attrs['inst_freq'], 0, 150)

    # RMS amplitude in a sliding 50-ms window
    win = 25  # samples
    rms = np.sqrt(uniform_filter(seis_data**2, size=(1, win)))
    attrs['rms_amplitude'] = rms

    return attrs


def semblance(seis_data, half_win=4, n_traces=3):
    """Multi-trace semblance (coherence) attribute."""
    N, T = seis_data.shape
    sem  = np.zeros((N, T))
    for tr in range(n_traces, N - n_traces):
        patch = seis_data[tr - n_traces : tr + n_traces + 1, :]   # 2n+1 traces
        for t in range(half_win, T - half_win):
            window = patch[:, t - half_win : t + half_win + 1]
            num    = np.sum(np.mean(window, axis=0)**2) * (2*n_traces + 1)
            den    = np.sum(window**2) + 1e-12
            sem[tr, t] = num / den
    return np.clip(sem, 0, 1)


attrs = compute_attributes(seismic_agc)
print("Computing semblance (this takes ~30 s) ...")
sem = semblance(seismic_agc, half_win=3, n_traces=2)
attrs['semblance'] = sem
print("Attributes computed.")

# ── Plot attributes ───────────────────────────────────────────
attr_list = [
    ('envelope',       'Instantaneous Envelope',   'hot',      None),
    ('cos_inst_phase', 'Cosine of Inst. Phase',    'gray',     (-1, 1)),
    ('inst_freq',      'Instantaneous Frequency',  'rainbow',  (0, 80)),
    ('rms_amplitude',  'RMS Amplitude',            'inferno',  None),
    ('semblance',      'Semblance (Coherence)',     'binary',   (0, 1)),
]

fig, axes = plt.subplots(1, 5, figsize=(22, 6), sharey=True)

ext = [0, x_axis[-1]/1000, TMAX*1000, 0]

for ax, (key, label, cmap, clim) in zip(axes, attr_list):
    d  = attrs[key]
    vm = (d.min(), d.max()) if clim is None else clim
    im = ax.imshow(d.T, aspect='auto', cmap=cmap,
                   vmin=vm[0], vmax=vm[1],
                   origin='upper', extent=ext)
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Distance [km]', fontsize=9)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(200))

axes[0].set_ylabel('TWT [ms]', fontsize=10)
fig.suptitle('Post-Stack Seismic Attributes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('seismic_attributes.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 9. Fault Detection via Edge Enhancement

We apply **variance** and **gradient** operators to detect faults from the semblance volume — a standard workflow in seismic interpretation software (e.g., Kingdom, OpendTect, Petrel).

In [ ]:
# ============================================================
#  CELL 11 — Fault Enhancement (Variance + Gradient)
# ============================================================

from scipy.ndimage import sobel, generic_gradient_magnitude, gaussian_gradient_magnitude

# Variance attribute: local deviation from mean
mean_local  = uniform_filter(seismic_agc, size=(5, 5))
var_attr    = uniform_filter((seismic_agc - mean_local)**2, size=(5, 5))

# Fault likelihood from semblance discontinuity
fault_like  = 1.0 - gaussian_filter(sem, sigma=1.2)

# Gradient magnitude of cosine phase — highlights reflector edges
grad_phase  = gaussian_gradient_magnitude(attrs['cos_inst_phase'], sigma=1.5)

# ── Side-by-side comparison ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True, sharex=True)

panels = [
    (seismic_agc, 'Seismic (AGC)', 'seismic', None),
    (fault_like,  'Fault Likelihood\n(1 − Semblance)', 'hot_r', (0, 0.8)),
    (grad_phase,  'Phase Gradient\n(Reflector Edges)', 'magma', None),
]

for ax, (d, title, cmap, clim) in zip(axes, panels):
    vm = (np.percentile(d, 1), np.percentile(d, 99)) if clim is None else clim
    im = ax.imshow(d.T, aspect='auto', cmap=cmap,
                   vmin=vm[0], vmax=vm[1],
                   origin='upper', extent=[0, x_axis[-1]/1000, TMAX*1000, 0])
    plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Distance [km]')
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.yaxis.set_major_locator(MultipleLocator(100))

axes[0].set_ylabel('TWT [ms]')
plt.suptitle('Fault Detection Attributes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fault_detection.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10. AVO (Amplitude Versus Offset) Modeling

**AVO** exploits the offset-dependence of reflectivity (Zoeppritz equations). We use the **Shuey 2-term** approximation:

$$R(\theta) \approx R_0 + G \sin^2\theta$$

where $R_0$ = zero-offset (acoustic) intercept and $G$ = AVO gradient. Gas sands show **Class III AVO** (bright, increasing amplitude with offset) — a key **DHI (Direct Hydrocarbon Indicator)**.

In [ ]:
# ============================================================
#  CELL 12 — AVO Modeling (Shuey 2-term Approximation)
# ============================================================

def shuey_2term(Vp1, Vs1, rho1, Vp2, Vs2, rho2):
    """Returns (R0, G) — AVO intercept and gradient."""
    dVp  = Vp2  - Vp1
    dVs  = Vs2  - Vs1
    drho = rho2 - rho1
    Vp   = 0.5*(Vp1 + Vp2)
    Vs   = 0.5*(Vs1 + Vs2)
    rho  = 0.5*(rho1 + rho2)

    R0   = 0.5 * (dVp/Vp + drho/rho)
    G    = 0.5 * dVp/Vp - 2*(Vs/Vp)**2 * (2*dVs/Vs + drho/rho)
    return R0, G


def avo_gather(R0, G, n_offsets=60, max_angle=40):
    """Simulate a CMP gather with AVO."""
    angles  = np.linspace(0, max_angle, n_offsets)   # degrees
    theta   = np.radians(angles)
    R_theta = R0 + G * np.sin(theta)**2
    return angles, R_theta


# Define interfaces of interest
AVO_cases = [
    ('Shale over Gas Sand (Class III)',
     'Shale_shallow', 'Sand_gas',  '#d62728'),
    ('Shale over Brine Sand (Class I)',
     'Shale_shallow', 'Sand_brine', '#1f77b4'),
    ('Shale over Limestone (Class IV)',
     'Shale_deep', 'Limestone', '#2ca02c'),
    ('Shale over Coal',
     'Shale_shallow', 'Coal', '#9467bd'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# AVO curves
for label, r_above, r_below, clr in AVO_cases:
    ra = ROCK_DB[r_above]
    rb = ROCK_DB[r_below]
    R0, G = shuey_2term(ra['Vp'], ra['Vs'], ra['rho'],
                         rb['Vp'], rb['Vs'], rb['rho'])
    angles, R = avo_gather(R0, G)
    axes[0].plot(angles, R, lw=2.2, color=clr, label=f"{label}\nR₀={R0:.3f}, G={G:.3f}")

axes[0].axhline(0, color='k', lw=0.8, ls='--')
axes[0].set_xlabel('Angle of Incidence [°]')
axes[0].set_ylabel('Reflection Coefficient')
axes[0].set_title('AVO Curves (Shuey 2-term)', fontweight='bold')
axes[0].legend(fontsize=8, loc='lower left')
axes[0].set_xlim(0, 40)
axes[0].grid(True, alpha=0.3)

# Intercept-Gradient crossplot
R0s, Gs, lbls, clrs = [], [], [], []
for label, r_above, r_below, clr in AVO_cases:
    ra = ROCK_DB[r_above]
    rb = ROCK_DB[r_below]
    R0, G = shuey_2term(ra['Vp'], ra['Vs'], ra['rho'],
                         rb['Vp'], rb['Vs'], rb['rho'])
    R0s.append(R0); Gs.append(G)
    lbls.append(label.split(' (')[0]); clrs.append(clr)

axes[1].scatter(R0s, Gs, c=clrs, s=150, zorder=5, edgecolors='k')
for R0, G, lbl in zip(R0s, Gs, lbls):
    axes[1].annotate(lbl, (R0, G), textcoords='offset points',
                     xytext=(8, 4), fontsize=9)
axes[1].axhline(0, color='k', lw=0.7, ls='--')
axes[1].axvline(0, color='k', lw=0.7, ls='--')

# AVO classification quadrants
axes[1].fill_betweenx([-0.5, 0], [-0.5, -0.5], [0, 0], alpha=0.06, color='red',    label='Class III (Gas sand)')
axes[1].fill_betweenx([0, 0.5],  [0, 0],        [0.5, 0.5], alpha=0.06, color='blue', label='Class I (Brine sand)')
axes[1].set_xlabel('AVO Intercept (R₀)')
axes[1].set_ylabel('AVO Gradient (G)')
axes[1].set_title('Intercept–Gradient Crossplot', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-0.5, 0.5)
axes[1].set_ylim(-0.5, 0.5)

plt.suptitle('AVO Analysis — Hydrocarbon Indicator', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('avo_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 11. CMP Gather Simulation (NMO Correction)

A **Common Midpoint (CMP) gather** assembles traces from different source-receiver offsets at the same midpoint. **Normal Moveout (NMO)** correction flattens the hyperbolic reflection events:

$$t(x) = \sqrt{t_0^2 + \frac{x^2}{V_{\text{NMO}}^2}}$$

In [ ]:
# ============================================================
#  CELL 13 — Synthetic CMP Gather with NMO
# ============================================================

def make_cmp_gather(RC_1d, wavelet, offsets, Vrms_func, dt, n_samp):
    """
    Build a CMP gather from 1D reflectivity using NMO traveltime.
    Vrms_func(t) → RMS velocity at time t [m/s]
    """
    gather = np.zeros((len(offsets), n_samp))
    t_vec  = np.arange(n_samp) * dt

    # Convolve reflectivity once for zero-offset trace
    conv_full = convolve(RC_1d, wavelet, mode='full')
    start     = len(wavelet) // 2
    trace0    = conv_full[start : start + n_samp]

    for i, x in enumerate(offsets):
        trace_nmo = np.zeros(n_samp)
        for j, t0 in enumerate(t_vec):
            if t0 < 1e-6: continue
            vnmo = Vrms_func(t0)
            t_nmo = np.sqrt(t0**2 + x**2 / vnmo**2)
            samp_nmo = t_nmo / dt
            if samp_nmo >= n_samp - 1: continue
            samp_lo  = int(samp_nmo)
            frac     = samp_nmo - samp_lo
            trace_nmo[j] = (1-frac)*trace0[samp_lo] + frac*trace0[samp_lo+1]
        # Mute above critical offset (NMO stretch)
        mute_samp = max(5, int(x / (offsets[-1]) * 30))
        trace_nmo[:mute_samp] = 0
        gather[i] = trace_nmo

    return gather


# Use trace 80 (anticline crest)
RC_1d_cmp = RC_grid_att[80]

# RMS velocity function (interval velocities from rock DB)
from scipy.interpolate import interp1d as scipy_interp1d
t_knots   = np.array([0.00, 0.14, 0.25, 0.40, 0.60, 0.80, 1.00]) 
v_knots   = np.array([1700, 2200, 2600, 3200, 4500, 5500, 5950], dtype=float)
Vrms_fn   = scipy_interp1d(t_knots, v_knots, kind='linear',
                            fill_value='extrapolate')

offsets_cmp = np.linspace(0, 3000, 60)   # 0 – 3 km

print("Building CMP gather ...")
gather_raw = make_cmp_gather(RC_1d_cmp, wavelet, offsets_cmp,
                              Vrms_fn, DT, N_SAMPLES)

# ── Plot gather ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)

vm = np.percentile(np.abs(gather_raw), 97)
axes[0].imshow(gather_raw.T, aspect='auto', cmap='seismic',
               vmin=-vm, vmax=vm, origin='upper',
               extent=[offsets_cmp[0], offsets_cmp[-1], TMAX*1000, 0])
axes[0].set_title('CMP Gather — Before NMO', fontweight='bold')
axes[0].set_xlabel('Offset [m]')
axes[0].set_ylabel('TWT [ms]')

# Overlay theoretical hyperbola at 400 ms
t0_hyp  = 0.40
v_hyp   = Vrms_fn(t0_hyp)
t_hyp   = np.sqrt(t0_hyp**2 + offsets_cmp**2 / v_hyp**2)
axes[0].plot(offsets_cmp, t_hyp*1000, 'y--', lw=2, label=f'NMO hyperbola\nt₀={t0_hyp*1000:.0f} ms')
axes[0].legend(fontsize=9, loc='lower right')

# Apply NMO correction
gather_nmo = np.zeros_like(gather_raw)
for i, x in enumerate(offsets_cmp):
    for j, t0 in enumerate(time_axis):
        if t0 < 1e-6: continue
        vnmo    = Vrms_fn(t0)
        t_moved = np.sqrt(t0**2 + x**2 / vnmo**2)
        samp_f  = t_moved / DT
        if samp_f >= N_SAMPLES - 1: continue
        s0 = int(samp_f); frac = samp_f - s0
        val = (1-frac)*gather_raw[i,s0] + frac*gather_raw[i, s0+1]
        gather_nmo[i, j] = val
    mute = max(5, int(x / offsets_cmp[-1] * 30))
    gather_nmo[i, :mute] = 0

axes[1].imshow(gather_nmo.T, aspect='auto', cmap='seismic',
               vmin=-vm, vmax=vm, origin='upper',
               extent=[offsets_cmp[0], offsets_cmp[-1], TMAX*1000, 0])
axes[1].set_title('CMP Gather — After NMO Correction', fontweight='bold')
axes[1].set_xlabel('Offset [m]')

plt.suptitle(f'CMP Gather Simulation (Trace 80 — Anticline Crest)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cmp_gather.png', bbox_inches='tight', dpi=150)
plt.show()
print("CMP gather done.")

---
## 12. Master Interpretation Panel

Final comprehensive display integrating the seismic section, geological model overlay, and attribute annotations.

In [ ]:
# ============================================================
#  CELL 14 — Master Interpretation Figure
# ============================================================

fig = plt.figure(figsize=(20, 20))
gs  = gridspec.GridSpec(3, 2, figure=fig,
                         height_ratios=[1.6, 1.0, 1.0],
                         hspace=0.35, wspace=0.25)

ext = [0, x_axis[-1]/1000, TMAX*1000, 0]

# ── (A) Main seismic section (top, full width) ────────────────
ax_main = fig.add_subplot(gs[0, :])
vm = np.percentile(np.abs(seismic_agc), 98)
ax_main.imshow(seismic_agc.T, aspect='auto', cmap='seismic',
               vmin=-vm, vmax=vm, origin='upper', extent=ext)

# Wiggle traces every 5 km
scale = 0.6 / (vm + 1e-12)
for tr in range(0, N_TRACES, 15):
    xp = tr * DX / 1000
    w  = seismic_agc[tr] * scale
    ax_main.fill_betweenx(time_axis*1000, xp, xp + w,
                           where=(seismic_agc[tr] > 0),
                           color='k', alpha=0.25, lw=0)
    ax_main.plot(xp + w, time_axis*1000, 'k-', lw=0.25, alpha=0.6)

# Geological annotation
annots = [
    (80*DX/1000,   220, '▲ ANTICLINE\n(structural high)',   'white', '#1f4e79'),
    (220*DX/1000,  480, '▼ SYNCLINE',                       'white', '#7b2d00'),
    (130*DX/1000,  680, '⚡ NORMAL\nFAULT',                  'yellow','#333333'),
    (210*DX/1000,  580, '↑ THRUST\nFAULT',                   'white', '#4a0080'),
    (255*DX/1000,  430, '🔷 SALT\nDIAPIR',                   'white', '#8b0000'),
    (60*DX/1000,   750, '≋ FRACTURE\nZONE',                  'white', '#006400'),
    (20*DX/1000,   340, '— UNCON-\nFORMITY',                 'white', '#4a3000'),
]
for xp, yp, txt, fg, bg in annots:
    ax_main.text(xp, yp, txt, ha='center', va='center',
                 fontsize=8.5, fontweight='bold', color=fg,
                 bbox=dict(boxstyle='round,pad=0.3', fc=bg, ec='white',
                           alpha=0.85, lw=1))

ax_main.set_xlabel('Distance [km]', fontsize=12)
ax_main.set_ylabel('TWT [ms]', fontsize=12)
ax_main.set_title('(A)  Synthetic Seismic Section — AGC + Wiggle Overlay',
                   fontsize=13, fontweight='bold', loc='left')
ax_main.xaxis.set_major_locator(MultipleLocator(1))
ax_main.yaxis.set_major_locator(MultipleLocator(100))

# ── (B) Geological model ──────────────────────────────────────
ax_geo = fig.add_subplot(gs[1, 0])
ax_geo.imshow(litho_grid.T, aspect='auto', cmap=litho_cmap, norm=norm,
              origin='upper', extent=ext)
ax_geo.set_title('(B)  Geological Model (Lithology)', fontweight='bold', loc='left')
ax_geo.set_xlabel('Distance [km]')
ax_geo.set_ylabel('TWT [ms]')
ax_geo.xaxis.set_major_locator(MultipleLocator(1))

# ── (C) Instantaneous envelope ────────────────────────────────
ax_env = fig.add_subplot(gs[1, 1])
im_env = ax_env.imshow(attrs['envelope'].T, aspect='auto', cmap='hot',
                        vmin=0, vmax=np.percentile(attrs['envelope'], 97),
                        origin='upper', extent=ext)
plt.colorbar(im_env, ax=ax_env, shrink=0.9)
ax_env.set_title('(C)  Instantaneous Envelope (Bright Spots)', fontweight='bold', loc='left')
ax_env.set_xlabel('Distance [km]')
ax_env.xaxis.set_major_locator(MultipleLocator(1))

# ── (D) Fault likelihood ──────────────────────────────────────
ax_flt = fig.add_subplot(gs[2, 0])
im_flt = ax_flt.imshow(fault_like.T, aspect='auto', cmap='hot_r',
                        vmin=0, vmax=0.8, origin='upper', extent=ext)
plt.colorbar(im_flt, ax=ax_flt, shrink=0.9)
ax_flt.set_title('(D)  Fault Likelihood (1−Semblance)', fontweight='bold', loc='left')
ax_flt.set_xlabel('Distance [km]')
ax_flt.set_ylabel('TWT [ms]')
ax_flt.xaxis.set_major_locator(MultipleLocator(1))

# ── (E) Velocity model ───────────────────────────────────────
ax_vel = fig.add_subplot(gs[2, 1])
im_vel = ax_vel.imshow(Vp_grid.T, aspect='auto', cmap='jet',
                        origin='upper', extent=ext)
plt.colorbar(im_vel, ax=ax_vel, shrink=0.9).set_label('Vp [m/s]')
ax_vel.set_title('(E)  P-wave Velocity Model', fontweight='bold', loc='left')
ax_vel.set_xlabel('Distance [km]')
ax_vel.xaxis.set_major_locator(MultipleLocator(1))

fig.suptitle(
    'SYNTHETIC SEISMIC DATA — MASTER INTERPRETATION PANEL\n'
    'Faults · Fractures · Folds · Unconformity · Salt · AVO',
    fontsize=15, fontweight='bold', y=1.01
)

plt.savefig('master_panel.png', bbox_inches='tight', dpi=150)
plt.show()
print("Master panel saved.")

---
## 13. Export Data

Save the synthetic dataset to **NumPy** `.npy` archives and a simple text header — ready for use in machine learning, seismic inversion, or geophysical algorithm benchmarking.

In [ ]:
# ============================================================
#  CELL 15 — Export Synthetic Dataset
# ============================================================

import os

EXPORT_DIR = './synthetic_seismic_dataset'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Arrays to save
exports = {
    'seismic_clean'      : seismic_clean,
    'seismic_noisy'      : seismic_noisy,
    'seismic_agc'        : seismic_agc,
    'litho_grid'         : litho_grid,
    'Vp_grid'            : Vp_grid,
    'AI_grid'            : AI_grid,
    'RC_grid'            : RC_grid,
    'attr_envelope'      : attrs['envelope'],
    'attr_inst_phase'    : attrs['inst_phase'],
    'attr_inst_freq'     : attrs['inst_freq'],
    'attr_semblance'     : sem,
    'attr_fault_like'    : fault_like,
    'time_axis_s'        : time_axis,
    'x_axis_m'           : x_axis,
    'ricker_wavelet_35hz': wavelet,
}

for name, arr in exports.items():
    fpath = os.path.join(EXPORT_DIR, f'{name}.npy')
    np.save(fpath, arr.astype(np.float32))

# Write metadata header
header = f"""# Synthetic Seismic Dataset — Metadata
# Generated by: Synthetic Seismic Generation Notebook
#
# Survey Geometry
N_TRACES   = {N_TRACES}
N_SAMPLES  = {N_SAMPLES}
DT_s       = {DT}
DX_m       = {DX}
TMAX_s     = {TMAX:.4f}
#
# Structures modeled
# - Anticline  (trace 80, amplitude 40 samples)
# - Syncline   (trace 220, amplitude 25 samples)
# - Normal fault   (trace 130, throw 22 samples)
# - Reverse fault  (trace 200, throw 15 samples)
# - Salt diapir    (trace 255, sample 215, radii 22×80)
# - Fracture zones (traces 60, 175, samples 150-420)
# - Unconformity   (traces 0-90, ~165 ms)
#
# Wavelet      : Ricker, f0=35 Hz, zero-phase
# Noise model  : Gaussian ({SNR_DB} dB SNR) + coherent ground roll
# Attenuation  : Q={Q_factor}, geometric spherical divergence
#
# Lithologies  : {LITHO_NAMES}
"""

with open(os.path.join(EXPORT_DIR, 'README.txt'), 'w') as f:
    f.write(header)

print(f"Dataset exported to: {EXPORT_DIR}/")
print(f"Files saved:")
for fn in os.listdir(EXPORT_DIR):
    sz = os.path.getsize(os.path.join(EXPORT_DIR, fn))
    print(f"  {fn:<40}  {sz/1024:.1f} KB")

---
## 14. Summary & References

### What was generated

| Component | Details |
|---|---|
| **Survey geometry** | 300 traces × 500 samples, dt=2 ms, dx=25 m, profile=7.5 km |
| **Geological structures** | Anticline, Syncline, Normal fault, Reverse/Thrust fault, Salt diapir, Fracture zones (×2), Unconformity |
| **Rock physics** | 16 lithologies with Vp, Vs, ρ from published values |
| **Forward model** | Convolutional model: Ricker 35 Hz wavelet, Q=80 attenuation, geometric spreading |
| **Noise** | 12 dB SNR Gaussian + coherent ground roll |
| **Attributes** | Envelope, Phase, Cosine-Phase, Inst. Frequency, RMS, Semblance, Fault Likelihood |
| **AVO** | Shuey 2-term model, Class I/III classification, R₀–G crossplot |
| **CMP gather** | NMO simulation with RMS velocity function, mute application |

### References

1. **Sheriff & Geldart (1995)** — *Exploration Seismology*, Cambridge University Press.
2. **Yilmaz (2001)** — *Seismic Data Analysis*, SEG.
3. **Shuey (1985)** — *A simplification of the Zoeppritz equations*, Geophysics 50(4).
4. **Castagna & Backus (1993)** — *Offset-Dependent Reflectivity*, SEG Investigations in Geophysics.
5. **Taner, Koehler & Sheriff (1979)** — *Complex seismic trace analysis*, Geophysics 44(6).
6. **Bahorich & Farmer (1995)** — *3D seismic discontinuity for faults and stratigraphic features*, The Leading Edge.
7. **Hampson & Russell (1984)** — *First-break interpretation using generalized linear inversion*, CSEG.

---
*Generated with NumPy · SciPy · Matplotlib · Python 3.x*